## Data Exploration & Cleaning
- Toronto Major Crime Indicators (MCI) : ~475k offence records (2014 - March 2026)
- Toronto Neighborhood Profiles : 2021 population per neighborhood

In [2]:
import pandas as pd
pd.set_option("display.max_columns", None)
print("pandas", pd.__version__)

pandas 3.0.3


In [3]:
# load crime data
from pathlib import Path
RAW = Path("..") / "data" / "raw"
print(RAW.resolve())
crime = pd.read_csv(RAW / "mci_occurrences.csv")
print("Rows, columns", crime.shape)

C:\Users\Jason\Downloads\Jason Qiu\Projects\Toronto Crime 0\data\raw
Rows, columns (474819, 29)


In [4]:
crime.head()

,OBJECTID,EVENT_UNIQUE_ID,REPORT_DATE,OCC_DATE,REPORT_YEAR,REPORT_MONTH,REPORT_DAY,REPORT_DOY,REPORT_DOW,REPORT_HOUR,OCC_YEAR,OCC_MONTH,OCC_DAY,OCC_DOY,OCC_DOW,OCC_HOUR,DIVISION,LOCATION_TYPE,PREMISES_TYPE,UCR_CODE,UCR_EXT,OFFENCE,CSI_CATEGORY,HOOD_158,NEIGHBOURHOOD_158,HOOD_140,NEIGHBOURHOOD_140,LONG_WGS84,LAT_WGS84
0,1,GO-20141260537,1388552400000,1388552400000,2014,January,1,1,Wednesday,4,2014.0,January,1.0,1.0,Wednesday,4,NSA,"Streets, Roads, Highways (Bicycle Path, Privat...",Outside,1430,100,Assault,Assault,NSA,NSA,NSA,NSA,0.000000,0.000000
1,2,GO-20141260912,1388552400000,1388552400000,2014,January,1,1,Wednesday,6,2014.0,January,1.0,1.0,Wednesday,4,D14,"Streets, Roads, Highways (Bicycle Path, Privat...",Outside,1610,100,Robbery With Weapon,Robbery,078,Kensington-Chinatown (78),078,Kensington-Chinatown (78),-79.401983,43.647598
2,3,GO-20141261368,1388552400000,1388552400000,2014,January,1,1,Wednesday,7,2014.0,January,1.0,1.0,Wednesday,7,D13,"Apartment (Rooming House, Condo)",Apartment,1430,100,Assault,Assault,107,Oakwood Village (107),107,Oakwood Village (107),-79.438030,43.685451
3,4,GO-20141260521,1388552400000,1388552400000,2014,January,1,1,Wednesday,2,2014.0,January,1.0,1.0,Wednesday,2,D14,Bar / Restaurant,Commercial,2120,200,B&E,Break and Enter,081,Trinity-Bellwoods (81),081,Trinity-Bellwoods (81),-79.412798,43.650928
4,5,GO-20141261370,1388552400000,1388552400000,2014,January,1,1,Wednesday,7,2014.0,January,1.0,1.0,Wednesday,7,D31,"Apartment (Rooming House, Condo)",Apartment,1430,100,Assault,Assault,027,York University Heights (27),027,York University Heights (27),-79.500577,43.762240


In [5]:
# 'REPORT_DATE is stored as' miliseconds, so we convert it into a proper datetime
crime["report_date"] = pd.to_datetime(crime["REPORT_DATE"], unit="ms")

years_match = (crime["report_date"].dt.year == crime["REPORT_YEAR"])
print(years_match)

0         True
1         True
2         True
3         True
4         True
          ... 
474814    True
474815    True
474816    True
474817    True
474818    True
Length: 474819, dtype: bool


In [6]:
# confirm
print("Date range:", crime["report_date"].min(), "to", crime["report_date"].max())
crime[["REPORT_DATE", "report_date", "REPORT_YEAR"]].head()

Date range: 2014-01-01 05:00:00 to 2026-03-31 04:00:00


,REPORT_DATE,report_date,REPORT_YEAR
0,1388552400000,2014-01-01 05:00:00,2014
1,1388552400000,2014-01-01 05:00:00,2014
2,1388552400000,2014-01-01 05:00:00,2014
3,1388552400000,2014-01-01 05:00:00,2014
4,1388552400000,2014-01-01 05:00:00,2014


In [7]:
# derive year, month, and quarter
crime["reported_year"] = crime["report_date"].dt.year
crime["reported_month"] = crime["report_date"].dt.month
crime["reported_quarter"] = crime["report_date"].dt.quarter
crime[["report_date", "reported_year", "reported_month", "reported_quarter"]].head()


,report_date,reported_year,reported_month,reported_quarter
0,2014-01-01 05:00:00,2014,1,1
1,2014-01-01 05:00:00,2014,1,1
2,2014-01-01 05:00:00,2014,1,1
3,2014-01-01 05:00:00,2014,1,1
4,2014-01-01 05:00:00,2014,1,1


In [8]:
crime.info()

<class 'pandas.DataFrame'>
RangeIndex: 474819 entries, 0 to 474818
Data columns (total 33 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   OBJECTID           474819 non-null  int64         
 1   EVENT_UNIQUE_ID    474819 non-null  str           
 2   REPORT_DATE        474819 non-null  int64         
 3   OCC_DATE           474819 non-null  int64         
 4   REPORT_YEAR        474819 non-null  int64         
 5   REPORT_MONTH       474819 non-null  str           
 6   REPORT_DAY         474819 non-null  int64         
 7   REPORT_DOY         474819 non-null  int64         
 8   REPORT_DOW         474819 non-null  str           
 9   REPORT_HOUR        474819 non-null  int64         
 10  OCC_YEAR           474664 non-null  float64       
 11  OCC_MONTH          474664 non-null  str           
 12  OCC_DAY            474664 non-null  float64       
 13  OCC_DOY            474664 non-null  float64       
 14 

7,313 records have `HOOD_158 == 'NSA'` ("Not in Specified Area"), the location couldn't be specified to a neighbourhood (their lat/long are `0.0`). Because every analysis here is based on neighbourhood and population, these rows can't be used and would distort totals. We drop them and report the count for transparency.


In [9]:
before = len(crime)
crime = crime[crime["HOOD_158"] != "NSA"].copy()
after = len(crime)
print(before - after)
print(crime["HOOD_158"].nunique())

7313
158



The raw file has 29 columns but we onlt use 11. 
So we keep and rename the identifiers, the reported date parts, the offence category, the 158-system neighbourhood, and coordinates, and drop the pre-split date parts (redundant), all occurrence-date fields (we chose reported date), and the 140-system columns (wrong boundary system).


In [10]:

keep = {
    "OBJECTID":          "incident_id", # unique per row (our per-offence key)
    "EVENT_UNIQUE_ID":   "event_id", # police occurrence # (repeats across offences)
    "report_date":       "reported_date",
    "reported_year":     "reported_year",
    "reported_month":    "reported_month",
    "reported_quarter":  "reported_quarter",
    "CSI_CATEGORY":      "offence_category",
    "HOOD_158":          "neighbourhood_id", # join key to the population table
    "NEIGHBOURHOOD_158": "neighbourhood_name",
    "LAT_WGS84":         "latitude",
    "LONG_WGS84":        "longitude",
}

crime_clean = crime[list(keep)].rename(columns=keep)

# two small type fixes
crime_clean["neighbourhood_id"] = crime_clean["neighbourhood_id"].astype(int) # '078' -> 78
crime_clean["reported_date"]    = crime_clean["reported_date"].dt.date # drop the meaningless time-of-day

print("shape:", crime_clean.shape)
crime_clean.head()


shape: (467506, 11)


,incident_id,event_id,reported_date,reported_year,reported_month,reported_quarter,offence_category,neighbourhood_id,neighbourhood_name,latitude,longitude
1,2,GO-20141260912,2014-01-01,2014,1,1,Robbery,78,Kensington-Chinatown (78),43.647598,-79.401983
2,3,GO-20141261368,2014-01-01,2014,1,1,Assault,107,Oakwood Village (107),43.685451,-79.438030
3,4,GO-20141260521,2014-01-01,2014,1,1,Break and Enter,81,Trinity-Bellwoods (81),43.650928,-79.412798
4,5,GO-20141261370,2014-01-01,2014,1,1,Assault,27,York University Heights (27),43.762240,-79.500577
5,6,GO-20141261478,2014-01-01,2014,1,1,Break and Enter,36,Newtonbrook West (36),43.792329,-79.421663


Notice that in the table above, the 'neighborhood_name' carries a redundant ID, e.g. `Kensington-Chinatown (78)`. Since the ID
already lives in `neighbourhood_id`, we strip the trailing `(NN)` for clean labels using regex.


In [11]:
crime_clean["neighbourhood_name"] = (
    crime_clean["neighbourhood_name"]
    .str.replace(r"\s*\(\d+\)$", "", regex=True)
)

crime_clean["neighbourhood_name"].head()


1       Kensington-Chinatown
2            Oakwood Village
3          Trinity-Bellwoods
4    York University Heights
5           Newtonbrook West
Name: neighbourhood_name, dtype: str


An important thing to consider is that the Neighbourhood Profiles file is transposed, characteristics run down the rows (2,604 of them) and neighbourhoods run across the columns. See the output fromt the cell below


In [12]:
profiles_raw = pd.read_excel(
    RAW / "neighbourhood_profiles.xlsx",
    sheet_name="hd2021_census_profile",
    header=None,
)
print("shape:", profiles_raw.shape)
profiles_raw.iloc[:6, :5]   


shape: (2604, 159)


,0,1,2,3,4
0,Neighbourhood Name,West Humber-Clairville,Mount Olive-Silverstone-Jamestown,Thistletown-Beaumond Heights,Rexdale-Kipling
1,Neighbourhood Number,1,2,3,4
2,TSNS 2020 Designation,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area,Not an NIA or Emerging Neighbourhood
3,Total - Age groups of the population - 25% sam...,33300,31345,9850,10375
4,0 to 14 years,4295,5690,1495,1575
5,0 to 4 years,1460,1650,505,505


We grab name (row 0), number (row 1), and population (row 3) from the above table and join them into a tidy table with one row per neighbourhood:
`neighbourhood_id`, `neighbourhood_name`, `population`.


In [13]:
profiles = pd.DataFrame({
    "neighbourhood_id":   pd.to_numeric(profiles_raw.iloc[1, 1:]).astype(int).values,
    "neighbourhood_name": profiles_raw.iloc[0, 1:].astype(str).str.strip().values,
    "population":         pd.to_numeric(profiles_raw.iloc[3, 1:]).astype(int).values,
})

print(profiles.shape)
print(profiles['population'].sum())
profiles.head()


(158, 3)
2761290


,neighbourhood_id,neighbourhood_name,population
0,1,West Humber-Clairville,33300
1,2,Mount Olive-Silverstone-Jamestown,31345
2,3,Thistletown-Beaumond Heights,9850
3,4,Rexdale-Kipling,10375
4,5,Elms-Old Rexdale,9355


Confirm if every ID in the crime table exists in the population table, and vice versa.


In [14]:
crime_ids = set(crime_clean["neighbourhood_id"].unique())
prof_ids = set(profiles["neighbourhood_id"].unique())

print(len(crime_ids))
print(len(prof_ids))

158
158


Preview neighbourhood safety rank by comparing incident/neighborhood and per-capita.
Using the last four complete years (2022–2025), we count incidents per neighbourhood,
compute incidents per 10,000 residents, rank both ways, and measure the rank shift.
- Large positive shift = looks safe by raw count, but high per-capita (small populations)
- Large negative shift = looks dangerous by count, but average per-capita (big populations)


In [15]:
recent = crime_clean[crime_clean["reported_year"].between(2022,2025)]

counts = (
    recent.groupby(["neighbourhood_id", "neighbourhood_name"]).size().reset_index(name="total_incidents")
)

ranked = counts.merge(profiles[["neighbourhood_id", "population"]], on="neighbourhood_id")
ranked["per_10k"] = (ranked["total_incidents"] * 10000 / ranked["population"])
ranked["rank_by_count"] = ranked["total_incidents"].rank(ascending=False, method="min").astype(int)
ranked["rank_by_rate"]  = ranked["per_10k"].rank(ascending=False, method="min").astype(int)
ranked["rank_shift"]    = ranked["rank_by_count"] - ranked["rank_by_rate"]

ranked.shape


(158, 8)

In [16]:
cols = ["neighbourhood_name", "population", "total_incidents", "per_10k", "rank_by_count", "rank_by_rate", "rank_shift"]

print("Low count, high per-capita (small, dense):")
display(ranked.sort_values("rank_shift", ascending=False).head()[cols])

print("High count, low per-capita (large population):")
display(ranked.sort_values("rank_shift").head()[cols])

Low count, high per-capita (small, dense):


,neighbourhood_name,population,total_incidents,per_10k,rank_by_count,rank_by_rate,rank_shift
100,Beechborough-Greenbrook,6260,524,837.060703,139,27,112
61,Playter Estates-Danforth,7680,699,910.156250,113,19,94
13,Kingsway South,8855,569,642.574816,135,53,82
37,Bridle Path-Sunnybrook-York Mills,8940,658,736.017897,118,39,79
70,University,6435,874,1358.197358,83,7,76


High count, low per-capita (large population):


,neighbourhood_name,population,total_incidents,per_10k,rank_by_count,rank_by_rate,rank_shift
115,Agincourt North,27540,986,358.024691,66,150,-84
129,Malvern East,26095,1005,385.131251,64,143,-79
105,Tam O'Shanter-Sullivan,27205,1117,410.586289,53,131,-78
31,Westminster-Branson,25705,951,369.966933,72,145,-73
124,Golfdale-Cedarbrae-Woburn,27085,1324,488.831457,41,107,-66


In [17]:
CLEANED = Path("..") / "data" / "cleaned"
CLEANED.mkdir(exist_ok=True)   

crime_clean.to_csv(CLEANED / "crime_incidents.csv", index=False)
profiles.to_csv(CLEANED / "neighbourhood_profiles.csv", index=False)

print(f"Wrote {len(crime_clean):,} crime rows -> crime_incidents.csv")
print(f"Wrote {len(profiles):,} neighbourhood rows -> neighbourhood_profiles.csv")


Wrote 467,506 crime rows -> crime_incidents.csv
Wrote 158 neighbourhood rows -> neighbourhood_profiles.csv
